<a href="https://colab.research.google.com/github/sabdaipry/fiuba-ceia-pln1/blob/main/Desafio_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [ ]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [ ]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [ ]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [ ]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [ ]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [ ]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [ ]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [ ]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palabra que no está en el documento.

In [ ]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [ ]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [ ]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [ ]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [ ]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [ ]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748])

Obtenemos los 5 documentos más similares:

In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [ ]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [ ]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [ ]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**

### **1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.




In [ ]:
# Importo la librería para elegir números al azar
import random

# Elijo 5 índices al azar de nuestro conjunto de entrenamiento
random.seed(42)
indices_azar = random.sample(range(X_train.shape[0]), 5)

for idx in indices_azar:
    # 1. Identifico la clase del documento original
    clase_original = newsgroups_train.target_names[y_train[idx]]
    print(f"--- Documento Original (Clase: {clase_original}) ---")

    # Imprimo los primeros 200 caracteres para entender de qué trata
    print(newsgroups_train.data[idx][:200].replace('\n', ' ') + "...\n")

    # 2. Calculo la similitud coseno contra TODOS los documentos de train
    cossim = cosine_similarity(X_train[idx], X_train)[0]

    # 3. Busco los índices de los 5 valores más altos.
    mostsim = np.argsort(cossim)[-6:-1][::-1]

    print("Los 5 documentos más similares son:")
    for i in mostsim:
        clase_sim = newsgroups_train.target_names[y_train[i]]
        similitud = cossim[i]
        print(f" - Similitud: {similitud:.4f} | Clase: {clase_sim}")
    print("\n" + "="*50 + "\n")

--- Documento Original (Clase: rec.sport.hockey) ---
This is a general question for US readers:  How extensive is the playoff coverage down there?  In Canada, it is almost impossible not to watch a series on TV (ie the only two series I have not had an ...

Los 5 documentos más similares son:
 - Similitud: 0.2250 | Clase: rec.sport.hockey
 - Similitud: 0.2174 | Clase: talk.politics.mideast
 - Similitud: 0.2164 | Clase: sci.crypt
 - Similitud: 0.2126 | Clase: alt.atheism
 - Similitud: 0.2111 | Clase: sci.crypt


--- Documento Original (Clase: comp.sys.mac.hardware) ---
  	I think this kind of comparison is pretty useless in general.  The processor is only good when a good computer is designed around it adn the computer is used in its designed purpose.  Comparing pro...

Los 5 documentos más similares son:
 - Similitud: 0.3542 | Clase: comp.sys.mac.hardware
 - Similitud: 0.3132 | Clase: comp.sys.mac.hardware
 - Similitud: 0.3041 | Clase: comp.sys.mac.hardware
 - Similitud: 0.2504 | Clase

Al analizar los 5 documentos aleatorios mediante Similitud Coseno sobre sus vectores TF-IDF, se pueden observar distintos comportamientos:

* Sensibilidad a palabras genéricas: El primer documento de hockey falló bastante (trajo política, ateísmo y criptografía). Probablemente se deba a que el texto usa palabras muy comunes ("question", "readers", "watch", "TV") y pocas específicas del deporte.
* Alta precisión en textos técnicos: En los documentos de `comp.sys.mac.hardware` y `comp.graphics`, el modelo acertó el 100% de los documentos similares. Esto puede ser porque usan vocabulario muy específico ("processor", "unix", "compile"). Esto indica que el vocabulario técnico o de nicho genera vectores TF-IDF muy distintivos que facilitan la agrupación correcta.
* Similitud semántica entre clases: En el texto de autos (`rec.autos`) el modelo trajo uno de motos (`rec.motorcycles`), y en el segundo de hockey trajo uno de béisbol. Esto tiene sentido porque comparten jerga entre sí. Hablan de "aceite" y "motores" o de "juegos", "fans" y "lesiones", por lo que matemáticamente se parecen mucho aunque la etiqueta exacta sea otra.
* Limitaciones por falta de contexto (Falsos positivos): En el último caso de hockey, el documento más similar es de religión cristiana (`soc.religion.christian`). Esto puede ser porque el texto original menciona a los jugadores "Peter", "John", "Mark" y "Dave". Este es un ejemplo de que el modelo no entiende el contexto (no sabe que son jugadores), solo cuenta palabras.


El método TF-IDF es efectivo para agrupar documentos por temática cuando existen palabras clave fuertes, pero al ser un enfoque estadístico de frecuencias, pierde el contexto semántico de las oraciones y puede ser engañado por nombres propios o vocabulario genérico.

---

### **2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.



In [ ]:
sim_matrix = cosine_similarity(X_test, X_train)

indices_mas_similares = np.argmax(sim_matrix, axis=1)

y_pred_zeroshot = y_train[indices_mas_similares]

f1_macro_zs = f1_score(y_test, y_pred_zeroshot, average='macro')
print(f"F1-Score Macro del clasificador por prototipos: {f1_macro_zs:.4f}")

F1-Score Macro del clasificador por prototipos: 0.5050


El modelo por asignación de similitud arrojó un F1-Score Macro de 0.5050. Este rendimiento del 50.5% supera ampliamente a la elección aleatoria (aprox. 5% para 20 clases). Esto demuestra que la vectorización TF-IDF genera un espacio latente donde los documentos de una misma temática tienden a agruparse y ser cercanos mediante la métrica del coseno. Sin embargo, se queda corto frente al 58.5% del Naive Bayes.

Este clasificador por prototipos actúa básicamente como un modelo de 1 Vecino Más Cercano (1-NN). Solo calcula distancias geométricas. Si un documento de prueba comparte muchas palabras irrelevantes o "ruido" con un documento de entrenamiento de otra clase, se equivocará fácilmente. No tiene un mecanismo para descartar variables poco útiles.

A diferencia de la similitud directa, `MultinomialNB` sí aprende de los datos. Calcula probabilidades condicionales, lo que le permite entender que ciertas palabras (como conectores o palabras genéricas) aparecen en muchas clases y, por ende, debe bajarles el peso, dándole más importancia a las palabras exclusivas de cada categoría.

---


### **3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.



In [ ]:
# 1. Mejoro el vectorizador (quito stopwords en inglés y filtro palabras extremas)
# min_df=5 ignoro palabras que aparecen en menos de 5 documentos...
tfidf_opt = TfidfVectorizer(stop_words='english', min_df=5, max_df=0.7)
X_train_opt = tfidf_opt.fit_transform(newsgroups_train.data)
X_test_opt = tfidf_opt.transform(newsgroups_test.data)

# 2. Pruebo ComplementNB
# El parámetro 'alpha' controla el suavizado.
# Primero probé con 0.1 y me dio un f1 de 0.6745
# Lo subo a 0.5 para probar...
clf_comp = ComplementNB(alpha=0.5)
clf_comp.fit(X_train_opt, y_train)

# 3. Predicción y Evaluación
y_pred_opt = clf_comp.predict(X_test_opt)
f1_macro_opt = f1_score(y_test, y_pred_opt, average='macro')

print(f"F1-Score Macro Optimizado con ComplementNB: {f1_macro_opt:.4f}")

F1-Score Macro Optimizado con ComplementNB: 0.6827


Al optimizar los hiperparámetros del vectorizador y cambiar el algoritmo, el modelo experimentó una mejora sustancial, elevando el F1-Score Macro del 0.5854 (baseline) a 0.6827. Las modificaciones que impulsaron este incremento fueron:

**Mejoras en el vectorizador:**
* Se agregó `stop_words='english'` para eliminar conectores y artículos vacíos de significado semántico.
* El parámetro `min_df=5` descartó errores ortográficos y términos extremadamente raros, reduciendo el sobreajuste.
* El parámetro `max_df=0.7` filtró aquellas palabras demasiado genéricas que aparecen en más del 70% del corpus y que no poseen poder discriminativo para separar las clases.

**Cambio de modelo a ComplementNB:**
* Se reemplazó `MultinomialNB` por `ComplementNB`. Este algoritmo está diseñado empíricamente para corregir sesgos cuando los conjuntos de datos están desbalanceados.
* Tras experimentar con el hiperparámetro de suavizado, un valor de `alpha=0.5` demostró ser el mejor balance para lidiar con probabilidades de términos poco frecuentes sin perder precisión, logrando el pico de rendimiento observado.

En conclusión, al depurar el espacio vectorial, el modelo Bayesiano pudo concentrar sus probabilidades condicionales en los términos verdaderamente representativos de cada uno de los 20 grupos de noticias.

---

### **4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.

In [ ]:
# Transpongo la matriz original
X_palabras = X_train.T

# Elijo las 5 palabras
palabras_a_probar = ['space', 'baseball', 'windows', 'god', 'car']

for palabra in palabras_a_probar:
    # Busco qué índice numérico tiene esta palabra en nuestro diccionario
    idx_palabra = tfidfvect.vocabulary_.get(palabra)

    if idx_palabra is not None:
        # Calculo la similitud de la palabra contra todas las demás palabras
        cossim_word = cosine_similarity(X_palabras[idx_palabra], X_palabras)[0]

        # Obtengo los índices de las 5 palabras más similares
        sim_indices = np.argsort(cossim_word)[-6:-1][::-1]

        print(f"Palabra objetivo: '{palabra.upper()}'")
        print("Palabras más similares:")
        for i in sim_indices:
            print(f" - {idx2word[i]} (Similitud: {cossim_word[i]:.4f})")
        print("-" * 30)

Palabra objetivo: 'SPACE'
Palabras más similares:
 - nasa (Similitud: 0.3304)
 - seds (Similitud: 0.2966)
 - shuttle (Similitud: 0.2928)
 - enfant (Similitud: 0.2803)
 - seti (Similitud: 0.2465)
------------------------------
Palabra objetivo: 'BASEBALL'
Palabras más similares:
 - tommorrow (Similitud: 0.1839)
 - football (Similitud: 0.1759)
 - penna (Similitud: 0.1734)
 - wintry (Similitud: 0.1690)
 - espn (Similitud: 0.1677)
------------------------------
Palabra objetivo: 'WINDOWS'
Palabras más similares:
 - dos (Similitud: 0.3037)
 - ms (Similitud: 0.2320)
 - microsoft (Similitud: 0.2219)
 - nt (Similitud: 0.2140)
 - for (Similitud: 0.1930)
------------------------------
Palabra objetivo: 'GOD'
Palabras más similares:
 - jesus (Similitud: 0.2688)
 - bible (Similitud: 0.2616)
 - that (Similitud: 0.2560)
 - existence (Similitud: 0.2548)
 - christ (Similitud: 0.2511)
------------------------------
Palabra objetivo: 'CAR'
Palabras más similares:
 - cars (Similitud: 0.1797)
 - criterium

Al transponer la matriz TF-IDF, el espacio vectorial se invierte: cada término se convierte en un vector cuyas dimensiones (features) son los documentos del corpus. La similitud coseno en este nuevo espacio nos permite identificar qué palabras tienden a co-ocurrir o aparecer bajo las mismas distribuciones de frecuencia en los textos.

A partir de los resultados obtenidos con las 5 palabras de prueba, se observa lo siguiente:

* El método logra agrupar vocabulario altamente específico de manera exitosa. "SPACE" recupera términos aeroespaciales ("nasa", "shuttle", "seti"), mientras que "WINDOWS" reconstruye el ecosistema de sistemas operativos de su contexto ("dos", "ms", "microsoft", "nt").
* El modelo es capaz de asociar una palabra con sus variantes gramaticales, marcas o actores relacionados. Por ejemplo, "CAR" se asocia con "cars" (plural), "civic" (modelo) y "dealer/owner" (actores). De igual forma, "BASEBALL" se relaciona con su entorno de transmisión ("espn") y otros deportes ("football").
* Esta técnica basada puramente en frecuencias globales puede arrojar ruido. Esto se evidencia en la aparición de palabras estructurales que superaron los filtros, como "that" (asociado a GOD) o "for" (asociado a WINDOWS). Estas palabras tienen alta similitud no por tener el mismo significado, sino porque estadísticamente co-ocurren mucho en los mismos documentos que las palabras objetivo.

En conclusión, transponer la matriz documento-término permite construir una representación vectorial primitiva de palabras. Aunque presenta ruidos propios de un método estadístico global, demuestra empíricamente que la semántica de una palabra puede inferirse analizando la distribución de los textos en los que aparece.
